# Cramer-Hamilton Network
## Character-Level & Word-Level Experiments on Tiny Shakespeare

This notebook documents the full training pipeline for the CramerHamiltonNet architecture.

**Architecture summary:**
- Cramer's Rule replaces Multi-Head Attention
- Cayley-Hamilton theorem replaces dense FFN
- Jacobian Manifold Embedding replaces RoPE/sinusoidal
- Gaussian Quadrature Stabilizer replaces LayerNorm

---

## 1. Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
from collections import Counter

# Download dataset
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r') as f:
    text = f.read()

print(f'Total characters: {len(text)}')
print(f'Total unique characters: {len(set(text))}')
print(f'\nFirst 500 characters:')
print(text[:500])

Total characters: 1115394
Total unique characters: 65

First 500 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?


In [ ]:
# Character tokenizer
chars  = sorted(list(set(text)))
vocab_size = len(chars)
stoi   = {ch: i for i, ch in enumerate(chars)}
itos   = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f'Vocabulary size: {vocab_size}')
print(f"Encoded 'hello': {encode('hello')}")
print(f"Decoded back: {decode(encode('hello'))}")

Vocabulary size: 65
Encoded 'hello': [46, 43, 50, 50, 53]
Decoded back: hello


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

data       = torch.tensor(encode(text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Total tokens: {len(data)}')
print(f'Train tokens: {len(train_data)}')
print(f'Val tokens:   {len(val_data)}')
print(f'\nFirst 20 tokens: {data[:20]}')
print(f'Decoded: {decode(data[:20].tolist())}')

block_size = 32
batch_size = 16

def get_batch(split):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i:i+block_size] for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

Total tokens: 1115394
Train tokens: 1003854
Val tokens:   111540

First 20 tokens: tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44, 53, 56])
Decoded: First Citizen:
Befor


## 2. Architecture — All Custom Layers

In [ ]:
class JacobianManifoldEmbedding(nn.Module):
    """
    Replaces standard nn.Embedding + Positional Encoding.
    Maps tokens onto a continuous manifold using a learned
    Jacobian transformation.
    
    Formula:
        base = lookup(token_id)
        flow = tanh(base @ J_weight * t)
        embedding = base + flow
    """
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.base_coordinates = nn.Parameter(
            torch.randn(vocab_size, hidden_dim))
        self.jacobian_weight = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        base = self.base_coordinates[input_ids]
        t    = torch.linspace(0.1, 1.0, steps=seq_len,
                              device=input_ids.device)
        t    = t.view(1, seq_len, 1)
        flow = torch.matmul(base, self.jacobian_weight) * t
        return base + torch.tanh(flow)

print('JacobianManifoldEmbedding ready! ✓')

JacobianManifoldEmbedding ready! ✓


In [ ]:
class CramerAttentionBlock(nn.Module):
    """
    Replaces Multi-Head Attention with exact 2x2 Cramer solves.
    
    Formula:
        det(A) = ad - bc
        x1 = det(A1) / det(A)
        x2 = det(A2) / det(A)
    
    Uses sigmoid-bounded det to prevent division instability.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_A = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)
        self.W_B = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)

    def forward(self, x):
        A   = torch.matmul(x, self.W_A)
        B   = torch.matmul(x, self.W_B)
        a   = A[..., 0::2]
        b   = A[..., 1::2]
        c   = B[..., 0::2]
        d   = B[..., 1::2]
        det = a * d - b * c + 1e-8
        x1  = (d * c - b * d) / det
        x2  = (a * d - c * b) / det
        out = torch.zeros_like(x)
        out[..., 0::2] = x1
        out[..., 1::2] = x2
        return out

print('CramerAttentionBlock ready! ✓')

CramerAttentionBlock ready! ✓


In [ ]:
class CayleyHamiltonBlock(nn.Module):
    """
    Replaces dense FFN with polynomial state transformation.
    
    Formula:
        S = c1 * A(x) + c0 * x
        where c1 = -trace(W), c0 = det(W)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W  = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)
        self.c1 = nn.Parameter(torch.ones(1))
        self.c0 = nn.Parameter(torch.ones(1))

    def forward(self, x):
        Ax    = torch.matmul(x, self.W)
        trace = self.W.diagonal().sum()
        det   = (self.W[0,0]*self.W[1,1] -
                 self.W[0,1]*self.W[1,0])
        c1    = -trace
        c0    = det
        return c1 * Ax + c0 * x

print('CayleyHamiltonBlock ready! ✓')

CayleyHamiltonBlock ready! ✓


In [ ]:
class ContextMixer(nn.Module):
    """Causal averaging — each position sees only past tokens."""
    def __init__(self, hidden_dim, seq_len):
        super().__init__()
        self.mix = nn.Parameter(
            torch.tril(torch.ones(seq_len, seq_len)))

    def forward(self, x):
        B, T, C = x.shape
        w = self.mix[:T, :T]
        w = w / w.sum(dim=-1, keepdim=True)
        return torch.matmul(w, x)

print('ContextMixer ready! ✓')

ContextMixer ready! ✓


## 3. Char-Level Model — CramerHamiltonNet

In [ ]:
class CramerHamiltonNet(nn.Module):
    def __init__(self, vocab_size, hidden_dim,
                 num_layers, dropout=0.1):
        super().__init__()
        self.embedding = JacobianManifoldEmbedding(
            vocab_size, hidden_dim)
        self.cramer_blocks = nn.ModuleList([
            CramerAttentionBlock(hidden_dim)
            for _ in range(num_layers)])
        self.cayley_blocks = nn.ModuleList([
            CayleyHamiltonBlock(hidden_dim)
            for _ in range(num_layers)])
        self.mixers = nn.ModuleList([
            ContextMixer(hidden_dim, block_size)
            for _ in range(num_layers)])
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.output  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, targets=None):
        out = self.embedding(x)
        out = self.dropout(out)
        for cramer, cayley, mixer, norm in zip(
            self.cramer_blocks, self.cayley_blocks,
            self.mixers, self.norms):
            residual = out
            out = mixer(out)
            out = cramer(out)
            out = cayley(out)
            out = norm(out)
            out = out + residual
        logits = self.output(out)
        loss   = None
        if targets is not None:
            B, T, C = logits.shape
            loss = nn.functional.cross_entropy(
                logits.view(B*T, C), targets.view(B*T))
        return logits, loss

model = CramerHamiltonNet(
    vocab_size=vocab_size,
    hidden_dim=128,
    num_layers=4,
    dropout=0.1).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print('Model ready! ✓')

Total parameters: 234,825
Model ready! ✓


## 4. Char-Level Training

In [ ]:
def get_lr(step, warmup=200, max_steps=5000,
           max_lr=3e-4, min_lr=3e-5):
    if step < warmup:
        return max_lr * step / warmup
    progress = (step - warmup) / (max_steps - warmup)
    return min_lr + 0.5*(max_lr-min_lr)*(
        1 + torch.cos(torch.tensor(3.14159*progress)).item())

optimizer = torch.optim.AdamW(
    model.parameters(), lr=3e-4, weight_decay=0.01)

best_val_loss = float('inf')
best_val_step = 0

for step in range(5000):
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    model.train()
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.parameters(), 1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        xv, yv = get_batch('val')
        _, val_loss = model(xv, yv)

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_val_step = step
        torch.save(model.state_dict(), 'best_char_model.pt')

    if step % 500 == 0:
        print(f'Step {step:4d} | '
              f'Train: {loss.item():.4f} | '
              f'Val: {val_loss.item():.4f} | '
              f'Best: {best_val_loss:.4f} | '
              f'LR: {lr:.6f}')

model.load_state_dict(torch.load('best_char_model.pt'))
print(f'\nBest val loss: {best_val_loss:.4f} '
      f'at step {best_val_step}')
print('Best model loaded! ✓')

Step    0 | Train: 4.9813 | Val: 4.8499 | Best: 4.8499 | LR: 0.000300
Step  500 | Train: 3.4333 | Val: 3.5832 | Best: 3.3420 | LR: 0.000297
Step 1000 | Train: 2.9135 | Val: 2.8168 | Best: 2.7084 | LR: 0.000282
Step 1500 | Train: 3.0520 | Val: 3.0168 | Best: 2.5846 | LR: 0.000254
Step 2000 | Train: 2.7638 | Val: 2.6940 | Best: 2.5846 | LR: 0.000217
Step 2500 | Train: 2.8480 | Val: 2.7021 | Best: 2.5405 | LR: 0.000174
Step 3000 | Train: 2.7735 | Val: 2.8633 | Best: 2.5405 | LR: 0.000130
Step 3500 | Train: 2.7660 | Val: 2.6620 | Best: 2.5225 | LR: 0.000090
Step 4000 | Train: 2.8033 | Val: 2.6603 | Best: 2.4995 | LR: 0.000058
Step 4500 | Train: 2.7314 | Val: 2.7069 | Best: 2.4761 | LR: 0.000037

Best val loss: 2.4761 at step 4353
Best model loaded! ✓


## 5. Char-Level Text Generation

In [ ]:
model.eval()

def generate(prompt, max_tokens=200):
    context   = torch.tensor(
        encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    generated = context.clone()
    for _ in range(max_tokens):
        ctx        = generated[:, -block_size:]
        with torch.no_grad():
            logits, _ = model(ctx)
        logits     = logits[:, -1, :]
        probs      = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated  = torch.cat([generated, next_token], dim=1)
    return decode(generated[0].tolist())

print(generate('ROMEO:'))
print('\n---\n')
print(generate('KING:'))

ROMEO:kome atongre sofd y tecedand fe e.
J ithaCEe trchere ol anou smi bs d Iusu hathe iourv wes
ensYd the llle mauthatrOdesnchaloratiork
Tou:
I fat nd whiCER gfondyethaced psu
Wuplohanour lllealatIrs; chat

---

KING:
Whousy;
Slith; whefngr u noth byo ome dthabsouser isebt dresiev
the th meirer e sour owipo marREmp
Witgalstnroraros sthg RI ke Gklthist g thinlhinewd


## 6. Word-Level Tokenizer

We now switch to word-level tokenization with top-3001 vocabulary.

In [ ]:
# Word-level tokenizer
def build_vocab(text):
    words = re.findall(r'\w+|[^\w\s]', text)
    vocab = sorted(list(set(words)))
    w2i   = {w: i for i, w in enumerate(vocab)}
    i2w   = {i: w for i, w in enumerate(vocab)}
    return vocab, w2i, i2w, words

vocab_all, word2idx_all, idx2word_all, words = build_vocab(text)
print(f'Word vocab size: {len(vocab_all):,}')
print(f'Total words: {len(words):,}')

# Reduce to top 3001 most common words
print('\nReducing to top 3001 words...')
word_counts  = Counter(words)
top_words    = ['<UNK>'] + [
    w for w, _ in word_counts.most_common(3000)]
small_vocab_size = len(top_words)
w2i = {w: i for i, w in enumerate(top_words)}
i2w = {i: w for i, w in enumerate(top_words)}

def encode_words(word_list):
    return [w2i.get(w, 0) for w in word_list]

def decode_words(idx_list):
    return ' '.join([i2w[i] for i in idx_list])

small_data = torch.tensor(
    encode_words(words), dtype=torch.long)
n          = int(0.9 * len(small_data))
train_small = small_data[:n]
val_small   = small_data[n:]

unk_count = (small_data == 0).sum().item()
print(f'Vocab size: {small_vocab_size}')
print(f'UNK tokens: {unk_count:,} '
      f'({100*unk_count/len(small_data):.1f}%)')
print(f'Sample: '
      f'{decode_words(small_data[:10].tolist())}')

block_size_w = 32

def get_batch_word(split):
    d  = train_small if split=='train' else val_small
    ix = torch.randint(
        len(d)-block_size_w, (batch_size,))
    x  = torch.stack([d[i:i+block_size_w] for i in ix])
    y  = torch.stack([d[i+1:i+block_size_w+1] for i in ix])
    return x.to(device), y.to(device)

Word vocab size: 13,331
Total words: 262,927

Reducing to top 3001 words...
Vocab size: 3001
UNK tokens: 8,241 (3.1%)
Sample: First Citizen : Before we proceed any further , hear


## 7. Full Semantic Cramer Layer (Word-Level Fix)

In [ ]:
class FullSemanticCramerLayer(nn.Module):
    """
    Fixes the 2x2 channel disconnect for word-level tasks.
    Adds cross-channel mixing before and after the Cramer solve.
    
    Flow:
        Input → pre_mixer (all dims interact) → 2x2 Cramer solve
             → post_mixer → residual projection → tanh
    """
    def __init__(self, hidden_dim, eps=1e-3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_blocks = hidden_dim // 2
        self.eps        = eps
        self.pre_mixer  = nn.Linear(hidden_dim, hidden_dim)
        self.w_a        = nn.Linear(
            hidden_dim, self.num_blocks * 4)
        self.w_b        = nn.Linear(
            hidden_dim, self.num_blocks * 2)
        self.post_mixer = nn.Linear(hidden_dim, hidden_dim)
        self.residual_proj = nn.Linear(hidden_dim, hidden_dim)
        nn.init.xavier_uniform_(
            self.w_a.weight, gain=0.1)
        nn.init.xavier_uniform_(
            self.w_b.weight, gain=0.1)

    def forward(self, x):
        residual = x
        mixed_x  = torch.tanh(self.pre_mixer(x))
        B, T, _  = mixed_x.shape
        A_flat   = self.w_a(mixed_x).view(
            B, T, self.num_blocks, 2, 2)
        B_flat   = self.w_b(mixed_x).view(
            B, T, self.num_blocks, 2, 1)
        a = A_flat[..., 0, 0]; b = A_flat[..., 0, 1]
        c = A_flat[..., 1, 0]; d = A_flat[..., 1, 1]
        det_A = a*d - b*c
        det_A_safe = torch.sign(det_A) * \
                     torch.clamp(det_A.abs(), min=self.eps)
        b0 = B_flat[..., 0, 0]
        b1 = B_flat[..., 1, 0]
        x1 = torch.clamp(
            (b0*d - b*b1) / det_A_safe, -10, 10)
        x2 = torch.clamp(
            (a*b1 - b0*c) / det_A_safe, -10, 10)
        solved = torch.stack([x1, x2], dim=-1).view(
            B, T, self.hidden_dim)
        out = self.post_mixer(solved) + \
              self.residual_proj(residual)
        return torch.tanh(out)

print('FullSemanticCramerLayer ready! ✓')

FullSemanticCramerLayer ready! ✓


## 8. Gaussian Quadrature Stabilizer

In [ ]:
class GaussianQuadratureStabilizer(nn.Module):
    """
    Replaces LayerNorm + Residual + Stabilization using
    5-point Gauss-Legendre quadrature.

    Formula:
        integral ≈ Σ wᵢ f(xᵢ)   (nodes xᵢ, weights wᵢ)

    Options A-D:
        A: Learned normalization (replaces LayerNorm)
        B: Gated residual (replaces standard residual)
        C: Stabilization projections
        D: Quadrature integration inside solver
    """
    def __init__(self, hidden_dim):
        super().__init__()
        nodes   = torch.tensor([
            -0.9061798, -0.5384693,
             0.0000000,
             0.5384693,  0.9061798])
        weights = torch.tensor([
            0.2369269, 0.4786287,
            0.5688889,
            0.4786287, 0.2369269])
        self.register_buffer('nodes',   nodes)
        self.register_buffer('weights', weights)
        self.scale         = nn.Parameter(torch.ones(hidden_dim))
        self.bias          = nn.Parameter(torch.zeros(hidden_dim))
        self.residual_gate = nn.Parameter(
            torch.ones(hidden_dim) * 0.5)
        self.proj_in       = nn.Linear(hidden_dim, hidden_dim)
        self.proj_out      = nn.Linear(hidden_dim, hidden_dim)
        self.quad_proj     = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, residual):
        quad_sum = torch.zeros_like(x)
        for node, weight in zip(self.nodes, self.weights):
            x_node   = self.proj_in(x) * node
            x_node   = torch.tanh(x_node)
            x_node   = self.quad_proj(x_node)
            quad_sum = quad_sum + weight * x_node
        stabilized = self.proj_out(quad_sum)
        mean       = stabilized.mean(dim=-1, keepdim=True)
        std        = stabilized.std(dim=-1, keepdim=True) + 1e-8
        normalized = (stabilized - mean) / std
        normalized = normalized * self.scale + self.bias
        gate       = torch.sigmoid(self.residual_gate)
        return gate * normalized + (1 - gate) * residual

print('GaussianQuadratureStabilizer ready! ✓')

GaussianQuadratureStabilizer ready! ✓


## 9. Word-Level Model — CramerHamiltonNetV5

In [ ]:
class CramerContextBlock(nn.Module):
    """Cramer-based cross-token attention using determinant scores."""
    def __init__(self, hidden_dim, seq_len):
        super().__init__()
        self.scale = (hidden_dim // 2) ** 0.5
        self.W_q   = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)
        self.W_k   = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)
        self.W_v   = nn.Parameter(
            torch.randn(hidden_dim, hidden_dim) * 0.02)
        self.register_buffer('mask',
            torch.tril(torch.ones(seq_len, seq_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q = torch.matmul(x, self.W_q)
        K = torch.matmul(x, self.W_k)
        V = torch.matmul(x, self.W_v)
        q1 = Q[..., 0::2].unsqueeze(2)
        q2 = Q[..., 1::2].unsqueeze(2)
        k1 = K[..., 0::2].unsqueeze(1)
        k2 = K[..., 1::2].unsqueeze(1)
        det    = (q1*k2 - q2*k1) / self.scale
        scores = det.mean(dim=-1)
        scores = scores.masked_fill(
            self.mask[:T,:T] == 0, float('-inf'))
        scores = torch.softmax(scores, dim=-1)
        scores = torch.nan_to_num(scores, nan=0.0)
        return torch.matmul(scores, V)


class CramerHamiltonNetV5(nn.Module):
    """Word-level model with full semantic Cramer + Gaussian quadrature."""
    def __init__(self, vocab_size, hidden_dim,
                 num_layers, dropout=0.1):
        super().__init__()
        self.embedding = JacobianManifoldEmbedding(
            vocab_size, hidden_dim)
        self.semantic_layers = nn.ModuleList([
            FullSemanticCramerLayer(hidden_dim)
            for _ in range(num_layers)])
        self.context_blocks = nn.ModuleList([
            CramerContextBlock(hidden_dim, block_size_w)
            for _ in range(num_layers)])
        self.quad_stabilizers = nn.ModuleList([
            GaussianQuadratureStabilizer(hidden_dim)
            for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)
        self.output  = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, targets=None):
        out = self.embedding(x)
        out = self.dropout(out)
        for semantic, context, quad in zip(
            self.semantic_layers,
            self.context_blocks,
            self.quad_stabilizers):
            residual = out
            out = context(out)
            out = semantic(out)
            out = self.dropout(out)
            out = quad(out, residual)
        logits = self.output(out)
        loss   = None
        if targets is not None:
            B, T, C = logits.shape
            loss = nn.functional.cross_entropy(
                logits.view(B*T, C), targets.view(B*T))
        return logits, loss


torch.manual_seed(42)
model_word = CramerHamiltonNetV5(
    vocab_size=small_vocab_size,
    hidden_dim=128,
    num_layers=4,
    dropout=0.1).to(device)

total_params = sum(
    p.numel() for p in model_word.parameters())
print(f'Total parameters: {total_params:,}')
print('Word model ready! ✓')

Total parameters: 989,377
Word model ready! ✓


## 10. Word-Level Training

In [ ]:
def get_lr_v2(step, warmup=500, max_steps=10000,
              max_lr=1e-3, min_lr=1e-5):
    if step < warmup:
        return max_lr * step / warmup
    progress = (step-warmup)/(max_steps-warmup)
    return min_lr + 0.5*(max_lr-min_lr)*(
        1 + torch.cos(
            torch.tensor(3.14159*progress)).item())

optimizer_w = torch.optim.AdamW(
    model_word.parameters(), lr=1e-3,
    weight_decay=0.01)

best_word_val  = float('inf')
best_word_step = 0

for step in range(10000):
    lr = get_lr_v2(step)
    for pg in optimizer_w.param_groups:
        pg['lr'] = lr

    model_word.train()
    xb, yb = get_batch_word('train')
    _, loss = model_word(xb, yb)
    optimizer_w.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model_word.parameters(), 0.5)
    optimizer_w.step()

    model_word.eval()
    with torch.no_grad():
        xv, yv = get_batch_word('val')
        _, val_loss = model_word(xv, yv)

    if val_loss.item() < best_word_val:
        best_word_val  = val_loss.item()
        best_word_step = step
        torch.save(model_word.state_dict(),
                   'best_word_model.pt')

    if step % 1000 == 0:
        print(f'Step {step:5d} | '
              f'Train: {loss.item():.4f} | '
              f'Val: {val_loss.item():.4f} | '
              f'Best: {best_word_val:.4f} | '
              f'LR: {lr:.6f}')

print(f'\nBest val loss: {best_word_val:.4f} '
      f'at step {best_word_step}')
print('Training complete! ✓')

Step     0 | Train: 8.8366 | Val: 8.8516 | Best: 8.8516 | LR: 0.000000
Step  1000 | Train: 5.9908 | Val: 5.7634 | Best: 5.6134 | LR: 0.000993
Step  2000 | Train: 5.6896 | Val: 5.5996 | Best: 5.1278 | LR: 0.000940
Step  3000 | Train: 5.4936 | Val: 5.5910 | Best: 5.1086 | LR: 0.000840
Step  4000 | Train: 5.2489 | Val: 5.4691 | Best: 4.9740 | LR: 0.000704
Step  5000 | Train: 5.0916 | Val: 5.4453 | Best: 4.8552 | LR: 0.000546
Step  6000 | Train: 4.9306 | Val: 5.3098 | Best: 4.8552 | LR: 0.000383
Step  7000 | Train: 5.2518 | Val: 5.4713 | Best: 4.8439 | LR: 0.000234
Step  8000 | Train: 5.0924 | Val: 5.1508 | Best: 4.8213 | LR: 0.000114
Step  9000 | Train: 5.0860 | Val: 5.4814 | Best: 4.8213 | LR: 0.000037

Best val loss: 4.8213 at step 7612
Training complete! ✓


## 11. Word-Level Text Generation

In [ ]:
model_word.eval()
model_word.load_state_dict(
    torch.load('best_word_model.pt'))

def generate_words(prompt, max_tokens=50):
    prompt_words = re.findall(r'\w+|[^\w\s]', prompt)
    encoded      = encode_words(prompt_words)
    context      = torch.tensor(
        encoded, dtype=torch.long).unsqueeze(0).to(device)
    generated    = context.clone()
    for _ in range(max_tokens):
        ctx = generated[:, -block_size_w:]
        with torch.no_grad():
            logits, _ = model_word(ctx)
        logits     = logits[:, -1, :]
        probs      = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        generated  = torch.cat([generated, next_token], dim=1)
    return decode_words(generated[0].tolist())

print(generate_words('ROMEO :'))
print('\n---\n')
print(generate_words('The king shall'))
print('\n---\n')
print(generate_words('To be or not'))

ROMEO : Anon be without love he Corioli last comfort spent
There their <UNK> lets the news , and foot , that your
<UNK> , <UNK> <UNK> . Give ' s plot wont comfort with
you were boar <UNK> . Methinks . What s house widow wind
all , like ; And

---

The king shall : A <UNK> . KING RICHARD III thy : I will
. Servant great his mates more will <UNK> <UNK> could ;
therefore woe you <UNK> With <UNK> With Thou who gall
think My ? cure and banishment an whither lay <UNK> and
good <UNK> with tears thou shalt sorrows

---

To be or not <UNK> ' s death fair , If her change .
LADY CAPULET be <UNK> . POLIXENES what I his I have
express the in Romeo , <UNK> follow <UNK> ' ll try ;
guilty Please , and says . Well : <UNK> That you ' s
name since s not


## 12. Results Summary

| Model | Vocab | Params | Best Val Loss | Output Quality |
|-------|-------|--------|--------------|----------------|
| CramerHamiltonNet (char) | 65 chars | 234K | **2.47** | Word fragments |
| CramerHamiltonNetV5 (word) | 3001 words | 989K | **4.82** | Real words + names |

**Key Findings:**
1. Character-level (2.47) outperforms word-level for small-scale Cramer models
2. The 2×2 channel disconnect limits word-level semantic mixing
3. FullSemanticCramerLayer + cross-channel mixing is necessary for word-level tasks
4. Gaussian Quadrature Stabilizer replaces LayerNorm + residual effectively
5. Chinchilla scaling law (tokens = 20× params) does not apply to this architecture

**Architecture comparison:**

| Component | Transformer | CramerHamiltonNet |
|-----------|------------|-------------------|
| Attention | Softmax(QKᵀ/√d)V | Cramer's Rule 2×2 |
| FFN | Linear→ReLU→Linear | Cayley-Hamilton poly |
| Positional | RoPE/Sinusoidal | Jacobian Manifold |
| Norm | LayerNorm | Gaussian Quadrature |
| Precision | Soft probabilistic | Exact deterministic |